In [ ]:
# Colab setup: clone the repo if needed, then install the extracted package.
import os
import pathlib
import subprocess
import sys

REPO_URL = "https://github.com/haydenyoungcs/gradient-ascent.git"
REPO_DIR = pathlib.Path("/content/gradient-ascent")

github_token = os.environ.get("GITHUB_TOKEN")
wandb_api_key = os.environ.get("WANDB_API_KEY")

try:
    from google.colab import userdata  # type: ignore

    if github_token is None:
        github_token = userdata.get("GITHUB_TOKEN")
    if wandb_api_key is None:
        wandb_api_key = userdata.get("WANDB_API_KEY")
except Exception:
    pass

cwd = pathlib.Path.cwd()
project_root = cwd if (cwd / "pyproject.toml").exists() else REPO_DIR

if not project_root.exists():
    if github_token:
        clone_url = REPO_URL.replace("https://", f"https://{github_token}@")
        subprocess.run(["git", "clone", clone_url], check=True)
    else:
        raise RuntimeError(
            "Repo checkout not found. For this private repo, add a Colab secret or env var named "
            "GITHUB_TOKEN, or clone the repo manually before running this notebook."
        )

if not (project_root / "pyproject.toml").exists():
    raise FileNotFoundError(f"Expected pyproject.toml under {project_root}, but it was not found.")

os.chdir(project_root)
print(f"Changed working directory to {project_root}")

repo_src = project_root / "src"
for path in [project_root, repo_src]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(project_root), "wandb", "pot", "scikit-learn"], check=True)

import wandb

if wandb_api_key:
    wandb.login(key=wandb_api_key, relogin=True)
else:
    print("wandb installed; set WANDB_API_KEY if you want online logging.")


## 1-4. Setup, training, and core model checkpoints

This section trains the original and retrained models, then runs five unlearning baselines: gradient ascent, SSD, SalUn, certified removal, and SCRUB. For GA, the preset below stays faithful to vanilla gradient ascent on the forget set, but is made slightly stronger in an easy-to-justify way: a modestly higher learning rate, more forget-set updates per epoch, BatchNorm buffers still frozen for clean evaluation, and a looser clip so updates are visible without becoming unstable.

It also saves simple baseline-specific diagnostics that are easy to explain in a dissertation. For GA, these are the mean forget-set cross-entropy and gradient norm at each unlearning step. For SCRUB, they are the student-teacher KL divergence on the forget and retain sets, the retain-set cross-entropy, and retain/forget accuracy. Together these show whether SCRUB is separating from the teacher on forgotten data while still staying close on retained data.

In [ ]:
import csv
import math
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import Image as IPyImage, display

from gradient_ascent.data import (
    default_num_workers,
    load_cifar10_datasets,
    make_forget_retain_subsets,
    make_loader,
    subset_for_class,
)
from gradient_ascent.experiments import CoreExperimentConfig, ensure_wandb_run, run_core_checkpoints
from gradient_ascent.models import Net
from gradient_ascent.training import build_amp_config, configure_runtime
from gradient_ascent.unlearning import GAConfig, SCRUBConfig

try:
    import wandb
except ImportError:
    wandb = None

warnings.filterwarnings("ignore", category=DeprecationWarning)

NUM_CLASSES = 10
OUT_DIR = "out"
resnet_model_depth = 50
ALGORITHM_ORDER = ["ga", "ssd", "salun", "certified", "scrub"]

configure_runtime()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
amp_config = build_amp_config(device)
USE_BF16 = amp_config.dtype == torch.bfloat16

trainset, testset = load_cifar10_datasets(root="./data")
use_cuda = device.type == "cuda"
num_workers = default_num_workers(use_cuda)
model_factory = lambda: Net(num_classes=NUM_CLASSES, pretrained=False, model_depth=resnet_model_depth).to(device)

# Keep the GA objective itself unchanged: we still do vanilla ascent on the
# forget set only. The preset is just a little less conservative so the effect
# is easier to study: slightly higher LR, more forget batches per epoch, BN
# buffers frozen for clean evaluation, and a looser clip for stability.
ga_config = GAConfig(
    lr=7e-6,
    epochs=12,
    max_batches_per_epoch=12,
    freeze_bn=True,
    grad_clip_norm=5.0,
)

# SCRUB stays close to the original paper's teacher-student intuition without
# adding too many moving parts: the original network is the frozen teacher, the
# student starts from the same checkpoint, retain data uses KL + CE, and forget
# data uses a negative KL term. The defaults below are deliberately stable and
# conservative rather than aggressively tuned.
scrub_config = SCRUBConfig(
    lr=5e-4,
    epochs=6,
    alpha=1.0,
    beta=1.0,
    gamma=1.0,
    temperature=2.0,
    weight_decay=1e-4,
    grad_clip_norm=1.0,
)

core_config = CoreExperimentConfig(
    num_classes=NUM_CLASSES,
    model_depth=resnet_model_depth,
    out_dir=OUT_DIR,
    num_epochs=10,
    unlearn_batch_size=512 if USE_BF16 else 256,
    ga_config=ga_config,
    scrub_config=scrub_config,
)

wandb_run = ensure_wandb_run(
    wandb,
    project="gradient-ascent",
    name="core-checkpoints",
    config={
        "target_label": core_config.target_label,
        "num_epochs": core_config.num_epochs,
        "batch_size": core_config.batch_size,
        "model_depth": core_config.model_depth,
        "ga_lr": core_config.ga_config.lr,
        "ga_epochs": core_config.ga_config.epochs,
        "ga_max_batches_per_epoch": core_config.ga_config.max_batches_per_epoch,
        "ga_grad_clip_norm": core_config.ga_config.grad_clip_norm,
        "scrub_lr": core_config.scrub_config.lr,
        "scrub_epochs": core_config.scrub_config.epochs,
        "scrub_alpha": core_config.scrub_config.alpha,
        "scrub_beta": core_config.scrub_config.beta,
        "scrub_gamma": core_config.scrub_config.gamma,
        "scrub_temperature": core_config.scrub_config.temperature,
    },
)

core_artifacts = run_core_checkpoints(
    model_factory=model_factory,
    trainset=trainset,
    testset=testset,
    device=device,
    use_cuda=use_cuda,
    num_workers=num_workers,
    config=core_config,
    wandb_run=wandb_run,
    wandb_module=wandb,
)

display(IPyImage(filename=core_artifacts.original_vs_retrain_plot_path))
for algorithm_key in ALGORITHM_ORDER:
    artifact = core_artifacts.algorithm_artifacts[algorithm_key]
    display(IPyImage(filename=artifact.classwise_percent_plot_path))
    display(IPyImage(filename=artifact.classwise_absolute_plot_path))

for key, value in core_artifacts.summary_metrics.items():
    print(f"{key}: {value:.3f}")


def compute_mean_forget_loss_and_grad_norm(model, loader, device):
    """Simple GA diagnostics on the forget set.

    We track two quantities that are easy to explain and interpret:
    1. mean forget-set cross-entropy, which should rise if GA is making the
       forget examples harder for the model;
    2. mean gradient norm of that forget loss, which shows whether the ascent
       signal is still active or has already flattened out.
    """
    criterion = nn.CrossEntropyLoss()
    model.eval()
    total_loss = 0.0
    total_samples = 0
    grad_norms = []
    params = [param for param in model.parameters() if param.requires_grad]

    for inputs, labels in loader:
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        model.zero_grad(set_to_none=True)
        logits = model(inputs)
        loss = criterion(logits, labels)
        grads = torch.autograd.grad(loss, params, retain_graph=False, create_graph=False)

        sq_norm = 0.0
        for grad in grads:
            sq_norm += grad.detach().float().pow(2).sum().item()
        grad_norms.append(math.sqrt(sq_norm))

        batch_size = int(labels.shape[0])
        total_loss += float(loss.item()) * batch_size
        total_samples += batch_size

    return {
        "forget_loss": total_loss / max(total_samples, 1),
        "forget_grad_norm": sum(grad_norms) / max(len(grad_norms), 1),
    }


forget_subset, _ = make_forget_retain_subsets(trainset, core_config.target_label)
diagnostic_loader = make_loader(
    forget_subset,
    batch_size=min(256, core_config.unlearn_batch_size or 256),
    shuffle=False,
    num_workers=num_workers,
    use_cuda=use_cuda,
)

snapshot_dir = Path(core_artifacts.algorithm_artifacts["ga"].snapshot_dir)
snapshot_paths = sorted(snapshot_dir.glob("epoch_*.pt"))
if not snapshot_paths:
    raise RuntimeError(f"No GA snapshots found under {snapshot_dir}")

ga_diagnostic_rows = []
for snapshot_path in snapshot_paths:
    epoch = int(snapshot_path.stem.split("_")[-1])
    model = model_factory()
    model.load_state_dict(torch.load(snapshot_path, map_location=device))
    diagnostics = compute_mean_forget_loss_and_grad_norm(model, diagnostic_loader, device)
    ga_diagnostic_rows.append({"epoch": epoch, **diagnostics})

csv_path = Path(OUT_DIR) / "ga_diagnostics.csv"
with csv_path.open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=["epoch", "forget_loss", "forget_grad_norm"])
    writer.writeheader()
    writer.writerows(ga_diagnostic_rows)

plot_path = Path(OUT_DIR) / "ga_diagnostics.png"
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
epochs = [row["epoch"] for row in ga_diagnostic_rows]
axes[0].plot(epochs, [row["forget_loss"] for row in ga_diagnostic_rows], marker="o", linewidth=2)
axes[0].set_title("GA forget-set loss")
axes[0].set_xlabel("Unlearning step")
axes[0].set_ylabel("Cross-entropy")
axes[0].grid(alpha=0.3)

axes[1].plot(epochs, [row["forget_grad_norm"] for row in ga_diagnostic_rows], marker="o", linewidth=2)
axes[1].set_title("GA forget-set gradient norm")
axes[1].set_xlabel("Unlearning step")
axes[1].set_ylabel("L2 norm")
axes[1].grid(alpha=0.3)

fig.savefig(plot_path, dpi=180)
plt.close(fig)

print(f"Saved GA diagnostics CSV to {csv_path}")
print(f"Saved GA diagnostics plot to {plot_path}")
display(IPyImage(filename=str(plot_path)))

if wandb_run is not None and wandb is not None:
    wandb_run.log(
        {
            "plots/ga_diagnostics": wandb.Image(str(plot_path)),
            "tables/ga_diagnostics": wandb.Table(
                data=[[row["epoch"], row["forget_loss"], row["forget_grad_norm"]] for row in ga_diagnostic_rows],
                columns=["epoch", "forget_loss", "forget_grad_norm"],
            ),
        }
    )


def temperature_kl(student_logits, teacher_logits, temperature):
    student_log_probs = F.log_softmax(student_logits.float() / temperature, dim=1)
    teacher_probs = F.softmax(teacher_logits.float() / temperature, dim=1)
    return F.kl_div(student_log_probs, teacher_probs, reduction="sum") * (temperature ** 2)


@torch.inference_mode()
def compute_scrub_snapshot_diagnostics(student_model, teacher_model, forget_loader, retain_loader, device, temperature):
    """Track the two sides of the SCRUB objective in plain language.

    Higher forget KL means the student is moving away from the teacher on the
    forgotten data. Lower retain KL and retain CE mean the student is still
    behaving similarly to the teacher and labels on retained data.
    """
    student_model.eval()
    teacher_model.eval()

    forget_kl_sum = 0.0
    forget_correct = 0
    forget_total = 0
    for inputs, labels in forget_loader:
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        student_logits = student_model(inputs)
        teacher_logits = teacher_model(inputs)
        forget_kl_sum += float(temperature_kl(student_logits, teacher_logits, temperature).item())
        forget_correct += int((student_logits.argmax(dim=1) == labels).sum().item())
        forget_total += int(labels.numel())

    retain_kl_sum = 0.0
    retain_ce_sum = 0.0
    retain_correct = 0
    retain_total = 0
    for inputs, labels in retain_loader:
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        student_logits = student_model(inputs)
        teacher_logits = teacher_model(inputs)
        retain_kl_sum += float(temperature_kl(student_logits, teacher_logits, temperature).item())
        retain_ce_sum += float(F.cross_entropy(student_logits.float(), labels, reduction="sum").item())
        retain_correct += int((student_logits.argmax(dim=1) == labels).sum().item())
        retain_total += int(labels.numel())

    return {
        "forget_kl": forget_kl_sum / max(forget_total, 1),
        "retain_kl": retain_kl_sum / max(retain_total, 1),
        "retain_ce": retain_ce_sum / max(retain_total, 1),
        "forget_acc": forget_correct / max(forget_total, 1),
        "retain_acc": retain_correct / max(retain_total, 1),
    }


retain_subset = subset_for_class(trainset, core_config.target_label, include=False)
retain_diagnostic_loader = make_loader(
    retain_subset,
    batch_size=min(256, core_config.scrub_config.batch_size or core_config.unlearn_batch_size or 256),
    shuffle=False,
    num_workers=num_workers,
    use_cuda=use_cuda,
)

teacher_model = model_factory()
teacher_model.load_state_dict(torch.load(core_artifacts.original_checkpoint_path, map_location=device))
teacher_model.eval()

scrub_snapshot_dir = Path(core_artifacts.algorithm_artifacts["scrub"].snapshot_dir)
scrub_snapshot_paths = sorted(scrub_snapshot_dir.glob("epoch_*.pt"))
if not scrub_snapshot_paths:
    raise RuntimeError(f"No SCRUB snapshots found under {scrub_snapshot_dir}")

scrub_diagnostic_rows = []
for snapshot_path in scrub_snapshot_paths:
    epoch = int(snapshot_path.stem.split("_")[-1])
    student_model = model_factory()
    student_model.load_state_dict(torch.load(snapshot_path, map_location=device))
    diagnostics = compute_scrub_snapshot_diagnostics(
        student_model,
        teacher_model,
        diagnostic_loader,
        retain_diagnostic_loader,
        device,
        core_config.scrub_config.temperature,
    )
    scrub_diagnostic_rows.append({"epoch": epoch, **diagnostics})

scrub_csv_path = Path(OUT_DIR) / "scrub_diagnostics.csv"
with scrub_csv_path.open("w", newline="") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=["epoch", "forget_kl", "retain_kl", "retain_ce", "forget_acc", "retain_acc"],
    )
    writer.writeheader()
    writer.writerows(scrub_diagnostic_rows)

scrub_plot_path = Path(OUT_DIR) / "scrub_diagnostics.png"
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
epochs = [row["epoch"] for row in scrub_diagnostic_rows]

axes[0, 0].plot(epochs, [row["forget_kl"] for row in scrub_diagnostic_rows], marker="o", linewidth=2)
axes[0, 0].set_title("SCRUB forget KL to teacher")
axes[0, 0].set_xlabel("Unlearning step")
axes[0, 0].set_ylabel("KL")
axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(epochs, [row["retain_kl"] for row in scrub_diagnostic_rows], marker="o", linewidth=2)
axes[0, 1].set_title("SCRUB retain KL to teacher")
axes[0, 1].set_xlabel("Unlearning step")
axes[0, 1].set_ylabel("KL")
axes[0, 1].grid(alpha=0.3)

axes[1, 0].plot(epochs, [row["retain_ce"] for row in scrub_diagnostic_rows], marker="o", linewidth=2)
axes[1, 0].set_title("SCRUB retain cross-entropy")
axes[1, 0].set_xlabel("Unlearning step")
axes[1, 0].set_ylabel("Cross-entropy")
axes[1, 0].grid(alpha=0.3)

axes[1, 1].plot(
    epochs,
    [100.0 * row["retain_acc"] for row in scrub_diagnostic_rows],
    marker="o",
    linewidth=2,
    label="retain accuracy",
)
axes[1, 1].plot(
    epochs,
    [100.0 * row["forget_acc"] for row in scrub_diagnostic_rows],
    marker="s",
    linewidth=2,
    label="forget accuracy",
)
axes[1, 1].set_title("SCRUB student accuracy on retain/forget sets")
axes[1, 1].set_xlabel("Unlearning step")
axes[1, 1].set_ylabel("Accuracy (%)")
axes[1, 1].grid(alpha=0.3)
axes[1, 1].legend()

fig.savefig(scrub_plot_path, dpi=180)
plt.close(fig)

print(f"Saved SCRUB diagnostics CSV to {scrub_csv_path}")
print(f"Saved SCRUB diagnostics plot to {scrub_plot_path}")
display(IPyImage(filename=str(scrub_plot_path)))

if wandb_run is not None and wandb is not None:
    wandb_run.log(
        {
            "plots/scrub_diagnostics": wandb.Image(str(scrub_plot_path)),
            "tables/scrub_diagnostics": wandb.Table(
                data=[
                    [
                        row["epoch"],
                        row["forget_kl"],
                        row["retain_kl"],
                        row["retain_ce"],
                        row["forget_acc"],
                        row["retain_acc"],
                    ]
                    for row in scrub_diagnostic_rows
                ],
                columns=["epoch", "forget_kl", "retain_kl", "retain_ce", "forget_acc", "retain_acc"],
            ),
        }
    )


## 5. Shared similarity helpers

Defines reusable activation extraction, metric evaluation, and plotting helpers used by downstream trajectory analysis. Standalone final pairwise snapshot generation is removed to avoid redundancy.

In [ ]:
# Shared similarity helpers are imported from the reusable package.
from gradient_ascent.models import DEFAULT_LAYER_NAMES
from gradient_ascent.similarity import (
    CCA,
    CKA,
    CosineSimilarity,
    EarthMoversDistance,
    EuclideanDistance,
    GromovWassersteinDistance,
    HIGHER_BETTER_METRICS,
    KLDivergence,
    LOWER_BETTER_METRICS,
    build_default_metrics,
    collect_model_activations,
    evaluate_pair_rows,
    plot_grouped_bars,
    transform_rows_for_plot,
)

layer_names = list(DEFAULT_LAYER_NAMES)
metrics = build_default_metrics()
higher_better_metrics = list(HIGHER_BETTER_METRICS)
lower_better_metrics = list(LOWER_BETTER_METRICS)
plot_metric_names = higher_better_metrics + lower_better_metrics

print("Shared similarity helpers and metrics loaded from gradient_ascent.similarity.")

## 6. Epoch-wise unlearning trajectories, MIA, and animations

Runs snapshot trajectories for all five baselines (GA, SSD, SalUn, certified removal, and SCRUB), computes MIA trajectories, and generates evolving similarity summaries and GIFs for both references: unlearned-vs-retrained and unlearned-vs-original.

In [ ]:
# Unlearning algorithm comparison trajectories across all five baselines.
# Computes artifact files in the package, then displays them compactly here.

from IPython.display import Image as IPyImage, display

from gradient_ascent.experiments import TrajectoryExperimentConfig, ensure_wandb_run, run_trajectory_analysis

if "collect_model_activations" not in globals() or "evaluate_pair_rows" not in globals() or "transform_rows_for_plot" not in globals():
    raise RuntimeError("Run the shared similarity helper cell before this cell.")

trajectory_config = TrajectoryExperimentConfig(
    num_classes=NUM_CLASSES,
    model_depth=resnet_model_depth,
    out_dir=OUT_DIR,
    trajectory_batch_size=512 if USE_BF16 else 256,
)

trajectory_artifacts = run_trajectory_analysis(
    model_factory=model_factory,
    trainset=trainset,
    testset=testset,
    device=device,
    use_cuda=use_cuda,
    num_workers=num_workers,
    config=trajectory_config,
    original_checkpoint_path=f"{OUT_DIR}/original_net.pt",
    retrained_checkpoint_path=f"{OUT_DIR}/retrained_from_scratch_net.pt",
    snapshot_dirs={
        algorithm_key: core_artifacts.algorithm_artifacts[algorithm_key].snapshot_dir
        for algorithm_key in ALGORITHM_ORDER
    },
    layer_names=layer_names,
    metric_names=plot_metric_names,
    lower_better_metrics=lower_better_metrics,
    activation_collector=lambda model, loader: collect_model_activations(
        model,
        loader,
        layer_names,
        device,
        max_batches=trajectory_config.max_batches_for_similarity,
    ),
    pair_evaluator=evaluate_pair_rows,
    transform_rows_for_plot=transform_rows_for_plot,
    wandb_run=ensure_wandb_run(wandb, project="gradient-ascent", name="unlearning-algorithm-comparison"),
    wandb_module=wandb,
)

for algorithm_key in ALGORITHM_ORDER:
    mia_artifact = trajectory_artifacts.mia_artifacts[algorithm_key]
    display(IPyImage(filename=mia_artifact.grid_plot_path))
    display(IPyImage(filename=mia_artifact.control_plot_path))
    for reference_key in ["retrained", "original"]:
        similarity_artifact = trajectory_artifacts.similarity_artifacts[algorithm_key][reference_key]
        display(IPyImage(filename=similarity_artifact.summary_plot_path))
        display(IPyImage(filename=similarity_artifact.gif_path))


## 7. Combined cross-algorithm trajectory comparison

Overlays all five baselines, including SCRUB, in a single consolidated similarity + MIA comparison figure.

In [ ]:
# Integrated combined comparison figure: similarity + MIA trajectories.
# Reads saved trajectory CSV artifacts and displays the final summary inline.

from IPython.display import Image as IPyImage, display

from gradient_ascent.experiments import CombinedComparisonConfig, ensure_wandb_run, save_combined_trajectory_comparison

combined_path = save_combined_trajectory_comparison(
    CombinedComparisonConfig(out_dir=OUT_DIR),
    wandb_run=ensure_wandb_run(wandb, project="gradient-ascent", name="unlearning-algorithm-comparison"),
    wandb_module=wandb,
)
print(f"Saved integrated comparison figure to {combined_path}")
display(IPyImage(filename=combined_path))
